In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering, DBSCAN, Birch
import warnings
warnings.filterwarnings("ignore")

In [2]:
file_path = './FCA_lung_0.01.h5ad'
adata = sc.read_h5ad(file_path)
sc.pp.subsample(adata, fraction=0.01)  
print(adata)
adata.obs['cell_type']
num_unique_cell_types = len(adata.obs['cell_type'].cat.categories)
num_unique_cell_types

AnnData object with n_obs × n_vars = 726 × 58521
    obs: 'cell_type', 'batch'
    var: 'n_cells'


9

In [3]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [4]:
sc.tl.leiden(adata, key_added='leiden')
print("Leiden labels:", adata.obs['leiden'].unique())
kmeans = KMeans(n_clusters=8, random_state=18)  
kmeans_labels = kmeans.fit_predict(adata.obsm['X_pca'])  
adata.obs['kmeans'] = kmeans_labels.astype(str)
print("KMeans labels:", adata.obs['kmeans'].unique())
spectral = SpectralClustering(n_clusters=8, random_state=18, affinity='nearest_neighbors')
spectral_labels = spectral.fit_predict(adata.obsm['X_pca'])
adata.obs['spectral'] = spectral_labels.astype(str)
print("Spectral labels:", adata.obs['spectral'].unique())

Leiden labels: ['2', '0', '4', '3', '1', ..., '9', '7', '5', '6', '10']
Length: 11
Categories (11, object): ['0', '1', '2', '3', ..., '7', '8', '9', '10']
KMeans labels: ['4' '3' '0' '1' '5' '2' '6' '7']
Spectral labels: ['0' '1' '4' '2' '3' '7' '6' '5']


In [5]:
cell_type_list = adata.obs['cell_type'].tolist()
unique_cell_types = adata.obs['cell_type'].unique().tolist()
print("Unique cell types:", unique_cell_types)
clustering_results = adata.obs[['leiden', 'kmeans', 'spectral']].copy()
print(clustering_results.head())
clustering_results.to_csv('clustering_results0313.csv', index=True)

Unique cell types: ['Stromal cells', 'Bronchiolar and alveolar epithelial cells', 'Lymphoid cells', 'Vascular endothelial cells', 'Ciliated epithelial cells', 'Neuroendocrine cells', 'Myeloid cells', 'Megakaryocytes', 'Lymphatic endothelial cells']
      leiden kmeans spectral
3590       2      4        0
23721      0      3        1
28544      4      0        4
57284      3      1        2
56489      2      4        1


In [6]:
from ghtree import process_clustering_network_part, process_clustering_network_all
process_clustering_network_all(
    csv_path='clustering_results.csv',
    output_gtree_path="Gtree0312.graphml",
    output_ttree_path="Ttree0312.graphml"
)

Reading clustering data
       leiden  kmeans  spectral  agglomerative  birch
3590        2       6         5              0      0
23721       0       6         6              0      0
28544       4       7         4              0      0
57284       3       0         2              0      0
56489       2       1         5              0      0
Processing column leiden


100%|█████████████████████████████████████████████████████████████████████████████████| 11/11 [00:00<00:00, 120.18it/s]


Processing column kmeans


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 54.93it/s]


Processing column spectral


...

Processing column birch


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 28.58it/s]


Calculating edge capacities


100%|██████████████████████████████████████████████████████████████████████| 259588/259588 [00:00<00:00, 528127.84it/s]


Calculating Gomory-Hu tree
Saving G
Graph G saved to Gtree0312.graphml
Saving T
Tree T saved to Ttree0312.graphml


In [9]:
from processpart import process_clustering_part
from processall import process_clustering_all
process_clustering_all("Ttree0312.graphml", "clustering_results0313.csv", "leiden", 1700, 1900,task=1)

cankao
10%: 1618.4
20%: 1644.6
25%: 1667.0
33%: 1734.0
50%: 1800.0
1700 1900
Cells with changed clusters:
Discrete cells: ['3590', '36577', '28544', '57284', '12766', '19864', '59587', '63934', '10291', '22902', '39664']
Final clustering results saved to 'final_clustering_results.csv'.
Summary of changes:
       leiden  optimized_leiden
3590        2                 2
23721       0                 0
28544       4                 4
57284       3                 3
56489       2                 2
